In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import UTCDateTime, Stream
from obspy.clients.fdsn import Client
from obspy.geodetics.base import gps2dist_azimuth
from obspy.signal.trigger import recursive_sta_lta, trigger_onset

# GUI bits
#import ipywidgets as widgets
#from IPython.display import display, clear_output

EARTHSCOPE = Client("IRIS")   # waveforms + StationXML
USGS = Client("USGS")               # AK events (ComCat/ANSS subset)

# Redoubt centre (60°29′06″N 152°44′31″W)
REDOUBT_LAT = 60 + 29/60 + 6/3600
REDOUBT_LON = -(152 + 44/60 + 31/3600)

RADIUS_DEG = 0.2
t0 = UTCDateTime("2009-03-20T00:00:00")
t1 = UTCDateTime("2009-03-23T00:00:00")  # exclusive end for 3 full days

ROOT = "redoubt_20090320_20090323"
os.makedirs(ROOT, exist_ok=True)

In [ ]:
inv = EARTHSCOPE.get_stations(
    latitude=REDOUBT_LAT, longitude=REDOUBT_LON,
    maxradius=RADIUS_DEG,
    starttime=t0, endtime=t1,
    level="response",
)

stationxml_path = os.path.join(ROOT, "Redoubt_0p2deg_StationXML.xml")
inv.write(stationxml_path, format="STATIONXML")
print("Wrote:", stationxml_path)
print(inv)

In [ ]:
cat = USGS.get_events(
    starttime=t0, endtime=t1,
    latitude=REDOUBT_LAT, longitude=REDOUBT_LON,
    maxradius=RADIUS_DEG,
    catalog="ak",
    includeallorigins=True,
    includeallmagnitudes=True,
    limit=20000
)

print("N events:", len(cat))
quakeml_path = os.path.join(ROOT, "AK_events_Redoubt_0p2deg_20090320_23.xml")
cat.write(quakeml_path, format="QUAKEML")
print("Wrote:", quakeml_path)

In [ ]:
rows = []
for ev in cat:
    org = ev.preferred_origin() or (ev.origins[0] if ev.origins else None)
    mag = ev.preferred_magnitude() or (ev.magnitudes[0] if ev.magnitudes else None)
    if org is None:
        continue
    rows.append(dict(
        time=org.time.datetime,
        lat=org.latitude,
        lon=org.longitude,
        depth_km=(org.depth or np.nan)/1000.0,
        mag=(mag.mag if mag else np.nan),
    ))

df = pd.DataFrame(rows).sort_values("time")
display(df.head())

In [ ]:
plt.figure()
plt.scatter(df["lon"], df["lat"], s=10)
plt.scatter([REDOUBT_LON], [REDOUBT_LAT], marker="*", s=150)
plt.xlabel("Longitude"); plt.ylabel("Latitude"); plt.title("AK events near Redoubt (0.2°)")
plt.show()

In [ ]:
plt.figure()
plt.scatter(df["time"], df["mag"], s=10)
plt.xlabel("Time"); plt.ylabel("Magnitude"); plt.title("Magnitude vs time (AK catalog subset)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
keep_prefixes = ("EH", "BH", "HH", "SH")

# build (net, sta, loc, cha) list from inventory
bulk_ids = []
for net in inv:
    for sta in net:
        for cha in sta:
            if not cha.code.startswith(keep_prefixes):
                continue
            if not cha.code.endswith("Z"):
                continue
            loc = cha.location_code or ""
            bulk_ids.append((net.code, sta.code, loc, cha.code))

bulk_ids = sorted(set(bulk_ids))
print("Channels:", len(bulk_ids))
bulk_ids[:10]

In [ ]:
def harmonize_resample_merge(st: Stream, day: UTCDateTime, day_end: UTCDateTime) -> Stream:
    st = st.copy()
    st.sort(keys=["network", "station", "location", "channel", "starttime"])
    st = st.split()

    out = Stream()
    for tr_id in sorted(set(tr.id for tr in st)):
        sts = st.select(id=tr_id).copy()
        srs = sorted(set(float(tr.stats.sampling_rate) for tr in sts))
        target_sr = max(srs)

        # interpolate to target_sr if needed
        for tr in sts:
            if float(tr.stats.sampling_rate) != target_sr:
                tr.interpolate(sampling_rate=target_sr, method="linear", starttime=tr.stats.starttime)

        # merge, pad gaps with NaN, then force exact day span
        sts.merge(method=1, fill_value=np.nan)
        sts.trim(day, day_end, pad=True, fill_value=np.nan)

        # you should now have exactly 1 trace per id
        out += sts

    out.sort(keys=["network", "station", "location", "channel", "starttime"])
    return out

In [ ]:
wf_dir = os.path.join(ROOT, "waveforms_daily")
os.makedirs(wf_dir, exist_ok=True)

day = t0
while day < t1:
    day_end = day + 24*3600

    st = EARTHSCOPE.get_waveforms_bulk(
        bulk=[(n, s, l, c, day, day_end) for (n, s, l, c) in bulk_ids],
        attach_response=False,
    )

    st2 = harmonize_resample_merge(st, day, day_end)

    # write per station per day
    for (net, sta) in sorted({(tr.stats.network, tr.stats.station) for tr in st2}):
        st_sta = st2.select(network=net, station=sta)
        if not st_sta:
            continue
        fname = f"{net}.{sta}.{day.date}.Z.mseed"
        st_sta.write(os.path.join(wf_dir, fname), format="MSEED")

    print("Done day:", day.date, "traces:", len(st2))
    day = day_end

In [ ]:
# index available stations from filenames
files = sorted([f for f in os.listdir(wf_dir) if f.endswith(".mseed")])
stations = sorted(set(f.split(".")[1] for f in files))

station_dd = widgets.Dropdown(options=stations, description="Station:")
date_dd = widgets.Dropdown(
    options=[UTCDateTime("2009-03-20").date, UTCDateTime("2009-03-21").date, UTCDateTime("2009-03-22").date],
    description="Day:"
)
btn_prev = widgets.Button(description="← Prev day")
btn_next = widgets.Button(description="Next day →")
out = widgets.Output()

def plot_day(station, day_date):
    from obspy import read
    day = UTCDateTime(str(day_date))
    fname_glob = f"*.{station}.{day.date}.Z.mseed"
    matches = [f for f in os.listdir(wf_dir) if f.endswith(f".{station}.{day.date}.Z.mseed")]
    if not matches:
        print("No file for", station, day.date)
        return
    st = Stream()
    for m in matches:
        st += read(os.path.join(wf_dir, m))

    # ensure it's clean and sorted for plotting
    st.sort(keys=["network","station","location","channel","starttime"])

    # Plot each Z trace as a dayplot
    for tr in st:
        tr.plot(type="dayplot", interval=60, title=f"{tr.id} {day.date}")

def redraw(*args):
    with out:
        clear_output(wait=True)
        plot_day(station_dd.value, date_dd.value)

def step_day(delta):
    dates = list(date_dd.options)
    i = dates.index(date_dd.value)
    j = max(0, min(len(dates)-1, i+delta))
    date_dd.value = dates[j]

btn_prev.on_click(lambda b: step_day(-1))
btn_next.on_click(lambda b: step_day(+1))
station_dd.observe(redraw, names="value")
date_dd.observe(redraw, names="value")

display(widgets.HBox([station_dd, btn_prev, btn_next, date_dd]))
display(out)
redraw()

In [ ]:
def sta_lta_triggers(tr, sta_s=1.0, lta_s=10.0, on=3.5, off=1.0, fmin=1.0, fmax=15.0):
    tr = tr.copy()
    tr.detrend("demean")
    tr.taper(0.01)
    tr.filter("bandpass", freqmin=fmin, freqmax=fmax, corners=4, zerophase=True)

    sr = tr.stats.sampling_rate
    cft = recursive_sta_lta(tr.data.astype(np.float64), int(sta_s * sr), int(lta_s * sr))
    on_off = trigger_onset(cft, on, off)
    # convert sample indices -> times
    picks = [(tr.stats.starttime + i/sr, tr.stats.starttime + j/sr) for i, j in on_off]
    return picks, cft

# Example: run on one station/day file and print trigger times
from obspy import read
example = sorted([f for f in os.listdir(wf_dir) if f.endswith(".mseed")])[0]
st = read(os.path.join(wf_dir, example))
tr = st[0]

picks, cft = sta_lta_triggers(tr)
print(tr.id, "N triggers:", len(picks))
print("First few:", picks[:5])

In [ ]:
from obspy.signal.trigger import coincidence_trigger

def associate_events_sta_lta(
    st_day,
    sta=1.0, lta=10.0,
    thr_on=3.5, thr_off=1.0,
    coincidence_sum=3,   # require at least 3 stations
    max_trigger_length=60,
    delete_long_trigger=True,
    trigger_off_extension=5.0,
    details=True,
):
    """
    Returns list of event dicts from coincidence_trigger.
    """
    trig = coincidence_trigger(
        "recstalta", thr_on, thr_off, st_day,
        sta=sta, lta=lta,
        coincidence_sum=coincidence_sum,
        max_trigger_length=max_trigger_length,
        delete_long_trigger=delete_long_trigger,
        trigger_off_extension=trigger_off_extension,
        details=details
    )
    return trig

In [ ]:
triglist = associate_events_sta_lta(st_day, coincidence_sum=3)
print("Associated events:", len(triglist))
print(triglist[0].keys())
print(triglist[0]["time"], triglist[0]["duration"], triglist[0]["stations"])

In [ ]:
from obspy import Stream

def extract_event_window(st_day, t_event, pre=20.0, post=80.0, pad=True, fill_value=np.nan):
    t1 = t_event - pre
    t2 = t_event + post
    st_win = st_day.copy().trim(t1, t2, pad=pad, fill_value=fill_value)
    # drop empty traces
    st_win = Stream([tr for tr in st_win if tr.stats.npts > 1])
    return st_win

In [ ]:
evt = triglist[0]
st_evt = extract_event_window(st_day, evt["time"], pre=20, post=80)
print(st_evt)

In [ ]:
from obspy.signal.trigger import pk_baer

def pick_pk_baer(tr, t_event, search_pre=10.0, search_post=20.0):
    """
    Returns pick time (UTCDateTime) or None.
    """
    tr2 = tr.copy().trim(t_event - search_pre, t_event + search_post, pad=True, fill_value=0.0)
    tr2.detrend("demean")
    tr2.taper(0.01)

    df = tr2.stats.sampling_rate
    # pk_baer uses integer params; tune these later
    p_pick_sample, _ = pk_baer(
        tr2.data.astype(np.float64),
        df,
        20,  # tdownmax
        60,  # tupevent
        7,   # thr1
        12,  # thr2
        100, # preset_len
        200  # p_dur
    )
    if p_pick_sample is None or p_pick_sample < 0:
        return None
    return tr2.stats.starttime + (p_pick_sample / df)

In [ ]:
from obspy.signal.trigger import ar_pick

def pick_ar_pick(tr_z, tr_n, tr_e, t_event, pre=5.0, post=30.0):
    """
    Returns (p_pick_time, s_pick_time) or (None, None)
    """
    z = tr_z.copy().trim(t_event - pre, t_event + post, pad=True, fill_value=0.0)
    n = tr_n.copy().trim(t_event - pre, t_event + post, pad=True, fill_value=0.0)
    e = tr_e.copy().trim(t_event - pre, t_event + post, pad=True, fill_value=0.0)

    for tr in (z, n, e):
        tr.detrend("demean")
        tr.taper(0.01)

    df = z.stats.sampling_rate
    # many parameters; these are “starter” values only
    p_idx, s_idx = ar_pick(
        z.data.astype(np.float64),
        n.data.astype(np.float64),
        e.data.astype(np.float64),
        df,
        1.0, 20.0,   # f1, f2 bandpass
        0.5, 1.0,    # lta_p, sta_p (s)
        1.0, 2.0,    # lta_s, sta_s (s)
        8.0,  # m_p
        4.0,  # m_s
        0.2,  # l_p
        0.2,  # l_s
        True  # pick S
    )
    p_time = z.stats.starttime + (p_idx / df) if p_idx and p_idx > 0 else None
    s_time = z.stats.starttime + (s_idx / df) if s_idx and s_idx > 0 else None
    return p_time, s_time

In [ ]:
from obspy.core.event import Catalog, Event, Origin, Pick, WaveformStreamID, Comment
from obspy.core.event.base import ResourceIdentifier

def build_catalog_from_triggers(triglist, st_day, day_label=""):
    cat = Catalog()
    for i, evt in enumerate(triglist):
        t_event = evt["time"]
        ev = Event(resource_id=ResourceIdentifier(f"smi:local/redoubt/{day_label}/evt{i:05d}"))

        # a very simple "origin": just store the trigger time for now
        origin = Origin(time=t_event)
        ev.origins = [origin]

        # store association info as comments
        ev.comments = [
            Comment(text=f"coincidence_sum={evt.get('coincidence_sum', '')}"),
            Comment(text=f"stations={','.join(evt.get('stations', []))}")
        ]

        # Make one pick per station on its Z trace (pk_baer baseline)
        for tr in st_day:
            if not tr.id.endswith("Z"):
                continue
            sta = tr.stats.station
            if sta not in set(evt.get("stations", [])):
                continue

            ptime = pick_pk_baer(tr, t_event)
            if ptime is None:
                continue

            wid = WaveformStreamID(
                network_code=tr.stats.network,
                station_code=tr.stats.station,
                location_code=tr.stats.location,
                channel_code=tr.stats.channel
            )
            pk = Pick(
                time=ptime,
                phase_hint="P",
                waveform_id=wid
            )
            ev.picks.append(pk)

        cat.events.append(ev)

    return cat